In [14]:
#import modules and load the data
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedKFold
from sklearn.model_selection import cross_val_score
from numpy import mean
from numpy import absolute
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GridSearchCV
import tensorflow as tf
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import KFold
import optuna

df = pd.read_csv("insurance.csv")
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [15]:
#Prepare the data
df = df.dropna() #remove missings
y = df['charges'] #select Target
X = df.drop('charges',axis=1) #Select features
X = pd.get_dummies(X, drop_first=True) #Turn Fuel_Type into dummy variables
columns = X.columns #create index with column names (needed for last step)
scaler = MinMaxScaler() #initiate the scaler
X = scaler.fit_transform(X) #scale the data
X = pd.DataFrame(X,columns=columns) #turn back into a dataframe

In [11]:
#Neural network with cross validation (first attempt)

#First define a function that contains the neural network model
def create_baseline():
    # create model
    nnmodel = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(X.shape[1],)),  # Input layer 
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(1)  # Output layer
    ])
    # Compile model
    nnmodel.compile(loss='mean_absolute_error', #loss is calculated with MAE
                    optimizer='adam', #type of gradient descent
                    metrics=['mae']) #use MAE to evaluate the model
    return nnmodel

# evaluate model 

#define the estimator
estimator = KerasRegressor(build_fn=create_baseline, #you have to use the kerasRegressor wrapper for nn models
                           epochs=200, batch_size=64, #200 epochs and 64 batches
                           verbose=0) #this will shorten the output text
#Define the kfold function
kfold = RepeatedKFold(n_splits=5, n_repeats=1, random_state=42) #I'm using just 5 splits and 1 repeat to save time, but more splits and repeats will be more accurate
#Generate the results
results = cross_val_score(estimator, X, y, cv=kfold,scoring='neg_mean_absolute_error')
#Print the results
print('the average prediction error is: %.0f' % mean(absolute(results)))

/usr/local/python/3.14.2/lib/python3.14/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


/usr/local/python/3.14.2/lib/python3.14/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


/usr/local/python/3.14.2/lib/python3.14/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/usr/local/python/3.14.2/lib/python3.14/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/usr/local/python/3.14.2/lib/python3.14/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


the average prediction error is: 1626


In [10]:
#Hyperparameter Tuning (warning, takes at least 15 minutes)

# Define the function to create the model
def build_model(n_layers=3, n_units=64, activation='softplus'): #start with some inital parameters
    nnmodel = tf.keras.models.Sequential() #define the model
    nnmodel.add(tf.keras.layers.Input(shape=(X.shape[1],)))  # Input layer
    for _ in range(n_layers): #create a for loop that goes through the number of layers, number of units, and activation types
        nnmodel.add(tf.keras.layers.Dense(n_units, activation=activation))
    nnmodel.add(tf.keras.layers.Dense(1))  # Output layer
    nnmodel.compile(loss='mean_absolute_error', optimizer='adam', metrics=['mae']) #compile the model and use mean absolute error as measure of fit
    return nnmodel

def objective(trial):
    # Suggest hyperparameters
    n_layers = trial.suggest_categorical('n_layers', [1, 2, 3, 4, 5])         # Number of hidden layers
    n_units = trial.suggest_categorical('n_units', [32, 64, 128, 256, 512])   # Number of nodes per hidden layer
    activation = trial.suggest_categorical('activation', ['relu', 'tanh', 'softplus', 'elu'])  # activation function
    epochs = trial.suggest_categorical('epochs', [50, 100, 150, 200])         # number of epochs
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])        # number of batches

    # Manual 5-fold CV (Keras models aren't natively supported by cross_val_score)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    mae_scores = []

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        nnmodel = build_model(n_layers=n_layers, n_units=n_units, activation=activation)
        nnmodel.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0)

        val_loss, val_mae = nnmodel.evaluate(X_val, y_val, verbose=0)
        mae_scores.append(val_mae)

    return np.mean(mae_scores)

# search across 10 different combinations
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=10)

print("Best parameters found: ", study.best_params) #print the best parameters

[I 2026-10-09 11:10:16,757] A new study created in memory with name: no-name-c955819e-f944-4a0a-9267-cc841415cd27
[I 2026-10-09 11:11:14,051] Trial 0 finished with value: 12808.448828125 and parameters: {'n_layers': 2, 'n_units': 128, 'activation': 'tanh', 'epochs': 200, 'batch_size': 64}. Best is trial 0 with value: 12808.448828125.
[I 2026-10-09 11:12:16,325] Trial 1 finished with value: 2195.6333251953124 and parameters: {'n_layers': 5, 'n_units': 512, 'activation': 'elu', 'epochs': 50, 'batch_size': 64}. Best is trial 1 with value: 2195.6333251953124.
[I 2026-10-09 11:13:21,729] Trial 2 finished with value: 3125.349609375 and parameters: {'n_layers': 2, 'n_units': 256, 'activation': 'relu', 'epochs': 200, 'batch_size': 64}. Best is trial 1 with value: 2195.6333251953124.
[I 2026-10-09 11:15:47,250] Trial 3 finished with value: 1635.393505859375 and parameters: {'n_layers': 5, 'n_units': 64, 'activation': 'relu', 'epochs': 200, 'batch_size': 16}. Best is trial 3 with value: 1635.393

Best parameters found:  {'n_layers': 5, 'n_units': 256, 'activation': 'relu', 'epochs': 200, 'batch_size': 64}


In [26]:
# Neural network trained on the entire dataset (no train/test split, no cross-validation)
 
def create_baseline():

    # create model

    nnmodel = tf.keras.models.Sequential([

        tf.keras.layers.Input(shape=(X.shape[1],)),  # Input layer

        tf.keras.layers.Dense(256, activation='relu'),  # hidden layer

        tf.keras.layers.Dense(256, activation='relu'),  # hidden layer

        tf.keras.layers.Dense(256, activation='relu'),  # hidden layer

        tf.keras.layers.Dense(256, activation='relu'),  # hidden layer

        tf.keras.layers.Dense(256, activation='relu'),  # hidden layer

        tf.keras.layers.Dense(1)  # Output layer

    ])
 
    # Compile model

    nnmodel.compile(

        loss='mean_absolute_error',

        optimizer='adam',

        metrics=['mae']

    )
 
    return nnmodel
 
 
# Build the model

final_model = create_baseline()
 
# Train on the full dataset

final_model.fit(

    X, y,

    epochs=200,

    batch_size=64,

    verbose=1

)
 

Epoch 1/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 13239.2676 - mae: 13239.2676
Epoch 2/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 10930.5293 - mae: 10930.5293
Epoch 3/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 7129.4551 - mae: 7129.4551
Epoch 4/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 5666.0996 - mae: 5666.0996
Epoch 5/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 4410.3208 - mae: 4410.3208
Epoch 6/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3709.0608 - mae: 3709.0608
Epoch 7/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 3405.1902 - mae: 3405.1902
Epoch 8/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3378.8777 - mae: 3378.8777
Epoch 9/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3268.4336 - mae: 3268.4336
Epoch 10/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3202.5513 - mae: 3202.5513
Epoch 11/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3170.2908 - mae: 3170.2908
Epoch 12/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/s

In [28]:
df_new = pd.read_csv("new_patients.csv")

x_new= pd.get_dummies(df_new, drop_first=True) #Turn Fuel_Type into dummy variables

columns= x_new.columns

x_new = x_new.reindex(columns=columns, fill_value=0) #make sure the new data has the same columns as the training data

predictions = final_model.predict(x_new) #make predictions on the new data

print(predictions) #print the predictions

ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 with name 'None' of layer 'dense_293' is incompatible with the layer: expected axis -1 of input shape to have value 8, but received input with shape (10, 7)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(10, 7), dtype=float32)
  • training=False
  • mask=None
  • kwargs=<class 'inspect._empty'>